# 03 — Scheme Effect Regression

Panel regression estimating how offensive/defensive scheme labels relate to
Ivy-only win percentage, controlling for school and year fixed effects.

Also splits by coach tenure phase (years 1-2 vs 3+) to test whether scheme
fit improves over time.


In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.models import run_scheme_regression, plot_scheme_coefs, scheme_by_tenure

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
master = pd.read_csv('../data/processed/master_labeled.csv')
print(master.shape)
master[['school','year','off_scheme','def_scheme','ivy_win_pct']].head()

## Full panel regression

In [ ]:
result = run_scheme_regression(master)
print(result['model_summary'].summary())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_scheme_coefs(result['coef_df'], ax=ax)
plt.savefig('../data/processed/scheme_coefs.png', dpi=150)
plt.show()
result['coef_df']

## Win % by scheme
Raw averages — before controlling for school/year.

In [ ]:
off_summary = (
    master.groupby('off_scheme')['ivy_win_pct']
    .agg(['mean','std','count'])
    .rename(columns={'mean':'avg_win_pct','std':'std','count':'n'})
    .sort_values('avg_win_pct', ascending=False)
)
print('=== Offensive Scheme Win % ===')
display(off_summary)

def_summary = (
    master.groupby('def_scheme')['ivy_win_pct']
    .agg(['mean','std','count'])
    .rename(columns={'mean':'avg_win_pct','std':'std','count':'n'})
    .sort_values('avg_win_pct', ascending=False)
)
print('=== Defensive Scheme Win % ===')
display(def_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
off_summary['avg_win_pct'].sort_values().plot.barh(ax=axes[0], color='#3498db', alpha=0.8)
axes[0].set_xlabel('Avg Ivy Win %')
axes[0].set_title('By Offensive Scheme')
def_summary['avg_win_pct'].sort_values().plot.barh(ax=axes[1], color='#e67e22', alpha=0.8)
axes[1].set_xlabel('Avg Ivy Win %')
axes[1].set_title('By Defensive Scheme')
plt.tight_layout()
plt.savefig('../data/processed/win_pct_by_scheme.png', dpi=150)
plt.show()

## Tenure analysis

Add coaching tenure data manually or via a CSV lookup table.
Format needed in `master`: columns `coach` and `coach_start_year`.

Example stub:

In [ ]:
# Load coach tenure lookup if it exists
import os
coach_path = '../data/raw/coaches.csv'
if os.path.exists(coach_path):
    coaches = pd.read_csv(coach_path)
    # Expected columns: school, year, coach, coach_start_year
    master_c = master.merge(coaches, on=['school','year'], how='left')
    tenure_table = scheme_by_tenure(master_c)
    display(tenure_table)
else:
    print("coaches.csv not found — create data/raw/coaches.csv with columns: school, year, coach, coach_start_year")
    print("Template saved to data/raw/coaches_template.csv")
    from src.scraper import IVY_SCHOOLS
    rows = []
    for school in IVY_SCHOOLS:
        for yr in range(2005, 2025):
            rows.append({'school': school, 'year': yr, 'coach': '', 'coach_start_year': ''})
    pd.DataFrame(rows).to_csv('../data/raw/coaches_template.csv', index=False)
    print('Template written. Fill in and rename to coaches.csv.')

## Style-score correlation with win %

In [ ]:
style_cols = ['pass_tendency','tempo','spread_factor','explosiveness','aggression','front_heaviness']
available = [c for c in style_cols if c in master.columns and master[c].notna().sum() > 5]

if available and 'ivy_win_pct' in master.columns:
    corrs = master[available + ['ivy_win_pct']].corr()['ivy_win_pct'].drop('ivy_win_pct')
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corrs]
    corrs.sort_values().plot.barh(ax=ax, color=colors, alpha=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Pearson r with Ivy Win %')
    ax.set_title('Style Score Correlations')
    plt.tight_layout()
    plt.savefig('../data/processed/style_correlations.png', dpi=150)
    plt.show()
    display(corrs.sort_values(ascending=False))
else:
    print('Insufficient style data for correlation analysis.')